# EX10: Implement PageRank 

## Overview
In this notebook, you will implement the **PageRank** algorithm.

You will break the work into small steps:
1. Load a graph
2. Construct the transition model
3. Implement PageRank via power iteration
4. Validate with sanity checks
5. Analyze results (top nodes, distribution, sensitivity)

You should have a more intuitive understanding of PageRank after completing the assignment.

You may use `numpy` and `matplotlib` for computations and plots.

**Do not** use `networkx.pagerank` or any other built-in PageRank implementation.


In [1]:
import numpy as np 
import networkx as nx 
import pylab as plot
import pandas as pd

## 1. Load the graph 


Included in the repository is a file `epl_final.csv`. This is a record of all matches in the English Premier League from 2000 - 2025. 

Load the csv with Pandas and transform the data into a graph. You should 
- Filter the dataframe to only matches that occured in the 2024/2025 season. 
- Add a vertex for every unique team
- Add a directed edge A -> B if team B won against team A. 

In [21]:
df = pd.read_csv("epl_final.csv")

season = df[df["Season"] == "2024/25"]

G = nx.DiGraph()

# Add every team as a vertex (even if it never won/lost)
G.add_nodes_from(pd.unique(season[["HomeTeam", "AwayTeam"]].values.ravel()))

for _, row in season.iterrows():
    home, away = row["HomeTeam"], row["AwayTeam"]
    homescore, awayscore = int(row["FullTimeHomeGoals"]), int(row["FullTimeAwayGoals"])
    if homescore > awayscore:
        G.add_edge(away, home)   # loser -> winner
    elif awayscore > homescore:
        G.add_edge(home, away)   # loser -> winner
    # draws: no edge

print(G.number_of_nodes(), "teams,", G.number_of_edges(), "edges")

20 teams, 198 edges


## 2. Build the transition matrix

Write a function which takes a networkX graph, and constructs the transition matrix from the graph. Recall that the transition matrix is defined as 
$$
\{P_{ij}\} = 
\begin{cases}
\frac{1}{outdeg(v_i)} & (i,j) \in E \\ 
\frac{1}{n} & outdeg(v_i) = 0 \\
0 & \text{otherwise}

\end{cases}
$$

In [22]:
# Your code here 
def buildTransitionMatrix(G):
    n = G.number_of_nodes()
    P = np.zeros((n, n))
    for ind, i in enumerate(G.nodes()):
        outdeg = G.out_degree(i)
        for ind2, j in enumerate(G.nodes()):
            if outdeg == 0: P[ind, ind2] = 1/n
            elif G.has_edge(i, j): P[ind, ind2] = 1/outdeg
    return P


In [23]:
P = buildTransitionMatrix(G)
print(P)
print("Row sums all 1:", np.allclose(P.sum(axis=1), 1))

[[0.         0.         0.         0.09090909 0.09090909 0.09090909
  0.         0.09090909 0.09090909 0.         0.09090909 0.09090909
  0.09090909 0.         0.09090909 0.09090909 0.         0.
  0.         0.09090909]
 [0.1        0.         0.         0.         0.1        0.1
  0.         0.1        0.         0.         0.         0.1
  0.1        0.1        0.         0.1        0.1        0.1
  0.         0.        ]
 [0.06666667 0.         0.         0.06666667 0.06666667 0.06666667
  0.06666667 0.06666667 0.06666667 0.06666667 0.06666667 0.06666667
  0.06666667 0.         0.06666667 0.06666667 0.         0.06666667
  0.         0.06666667]
 [0.         0.33333333 0.         0.         0.         0.
  0.         0.         0.         0.         0.33333333 0.
  0.         0.         0.         0.         0.33333333 0.
  0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.33333333 0.         0.         0.33333333
  0.33333

# 2.5 Build the full transition matrix (with teleportation)

Recall the matrix actually used in PageRank is the following 
$$
P' = dP + (1-d) \frac{1}{n}J
$$
Where $J$ is the $n \times n$ matrix of all ones. And $d$ is a *damping factor*, a hyperparameter which controls the probablility a teleportation occurs.

In [24]:
# Your code here
def buildFullMatrix(G,d=0.85):
    P = buildTransitionMatrix(G)
    n = G.number_of_nodes()
    return (d * P) + ((1-d) * (1/n) * np.ones((n,n)))


print(buildFullMatrix(G, d=0.85))

[[0.0075     0.0075     0.0075     0.08477273 0.08477273 0.08477273
  0.0075     0.08477273 0.08477273 0.0075     0.08477273 0.08477273
  0.08477273 0.0075     0.08477273 0.08477273 0.0075     0.0075
  0.0075     0.08477273]
 [0.0925     0.0075     0.0075     0.0075     0.0925     0.0925
  0.0075     0.0925     0.0075     0.0075     0.0075     0.0925
  0.0925     0.0925     0.0075     0.0925     0.0925     0.0925
  0.0075     0.0075    ]
 [0.06416667 0.0075     0.0075     0.06416667 0.06416667 0.06416667
  0.06416667 0.06416667 0.06416667 0.06416667 0.06416667 0.06416667
  0.06416667 0.0075     0.06416667 0.06416667 0.0075     0.06416667
  0.0075     0.06416667]
 [0.0075     0.29083333 0.0075     0.0075     0.0075     0.0075
  0.0075     0.0075     0.0075     0.0075     0.29083333 0.0075
  0.0075     0.0075     0.0075     0.0075     0.29083333 0.0075
  0.0075     0.0075    ]
 [0.0075     0.0075     0.0075     0.0075     0.0075     0.0075
  0.0075     0.0075     0.29083333 0.0075     0.

# 3. PageRank Dense Matrix
Write a function which computes the PageRank of the graph explicitly, through iteration. 

Recall that if the PageRank approximation at step $t$ is $r_t$, 
$$
r_{t+1} = (P')^T r_t
$$

Update $r$ each iteration until either 
- $||r_{t+1} - r_t ||_1 \leq tol$ 
- $t > max\_iter$

In [ ]:
def PageRank(G, d=0.85, tol=1e-6, max_iter=200):
    n = G.number_of_nodes()
    r = (1/n) * np.ones(n)
    P = buildFullMatrix(G, d)
    for t in range(max_iter):
        r_next = P.T @ r
        if np.abs(r_next - r).sum() <= tol:
            return r_next
        r = r_next
    return r



r = PageRank(G)

print(f"Sum of entries is {np.sum(r)}, should be close to one.")

## 3.5 Check against networkx implementation


In [ ]:
r_networkx = nx.pagerank(G)
r_networkx = np.array([r_networkx[v] for v in G.nodes()])  # dict -> array in the same node order

np.all(np.isclose(r_networkx,r))

## 4. PageRank Sparse Iteration
Of course, the real cleverness in PageRank is the insight that you don't need to explictly construct the full matrix, by exploiting this is a graph problem. Use the starter code below to complete a function which performs a single step of PageRank without the dense matrix.

In [34]:
def pagerank_step(G,r,d=0.85): 
    n = G.number_of_nodes()
    r_next = np.zeros(n,dtype=np.float64)
    index = {v: i for i, v in enumerate(G.nodes())}

    #Fill all entries in r_next with the teleportation weights
    r_next += (1-d) / n

    #Then, iterate over all vertices 
    #Propagate each vertex influence to its out-neighbors. 
    #Vertices with outdegree 0 should be handled differently. 
    for i, v in enumerate(G.nodes()):
        outdeg = G.out_degree(v)
        if outdeg == 0:
            r_next += d * r[i] / n          # dangling node: spread evenly over all vertices
        else:
            for u in G.successors(v):
                r_next[index[u]] += d * r[i] / outdeg

    return r_next

## 4.5 PageRank Sparse Power Iteration
Again, write a function which iteratively updates a vector until convergence

In [ ]:
def PageRank_Sparse(G,d=0.85, tol=1e-6, max_iter=200):
    n = G.number_of_nodes()
    r = (1/n) * np.ones(n)
    for t in range(max_iter):
        r_next = pagerank_step(G, r, d)
        if np.abs(r_next - r).sum() <= tol:
            return r_next
        r = r_next
    return r

Again, do a small sanity check against your previous and networkx implmentation.

In [ ]:
# your code here
r_sparse = PageRank_Sparse(G)

print(f"Sum of entries is {np.sum(r_sparse)}, should be close to one.")
print("Sparse vs dense:   ", np.all(np.isclose(r_sparse, r)))
print("Sparse vs networkx:", np.all(np.isclose(r_sparse, r_networkx)))

## Sparse vs. Full 
You should conduct the below results on the `Wiki-Vote.txt` data (edgelist). This data includes a directed edge A -> B if user A voted to promote user B to an admin role on Wikipedia.

You can verify that such a large graph will be very slow (or even run out of memory) on the full dense matrix implementation.

## 5. Results: top nodes by PageRank

### Task
- Print the top-10 nodes by PageRank score
- Include their scores

Write a short 2-3 sentences on how these scores can be interpreted in your data.


**your interpretation**

## 6. Light analysis

### 6.1 Distribution of PageRank scores
Plot a histogram of PageRank scores.

Questions to answer:
- Is the distribution highly skewed?
- Do most nodes have very small score?


### 6.2 Compare damping factors

Run PageRank with different damping factors and compare:
- `d = 0.70`, `0.85`, `0.95`

Tasks:
1. Compute rank vectors for each `d`
2. Compare the **top-10 overlap** between settings
3. Compute L1 distances between the resulting rank vectors

Answer briefly:
- Which nodes are stable across damping factors?
- Does higher `d` concentrate more probability on a smaller set of nodes?


### 6.3 Relationship with out-degree

Compute correlation between PageRank and out-degree.

Questions:
- Are high PageRank nodes necessarily high out-degree nodes?
- What do you observe in the scatter plot?
